In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
pd.options.display.float_format = '{:,.2f}'.format 
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

In [ ]:
data = pd.read_csv('cost_revenue_dirty.csv')
data.shape
data.sample(5)
data.isna().values.any()
data.duplicated().values.any()

In [ ]:
chars_to_remove = [',', '$']
columns_to_clean = ['USD_Production_Budget', 'USD_Worldwide_Gross', 'USD_Domestic_Gross']
for col in columns_to_clean:
    for char in chars_to_remove:
        data[col] = data[col].astype(str).str.replace(char, '', regex=False)
    data[col] = pd.to_numeric(data[col])

In [ ]:
data.head()
data['Release_Date'] = pd.to_datetime(data['Release_Date'])
data.head()
data.info()
data.describe()

In [ ]:
data[data.USD_Production_Budget == 1100]
zero_worldwide = data[data.USD_Worldwide_Gross == 0]
international = data.query('USD_Domestic_Gross == 0 and USD_Worldwide_Gross != 0')

In [ ]:
scrape_date = pd.Timestamp('2018-5-1')
future_releases = data[data.Release_Date >= scrape_date]
data_clean = data.drop(future_releases.index)

In [ ]:
money_losing = data_clean.query('USD_Production_Budget > USD_Worldwide_Gross')
money_losing.shape[0] / data_clean.shape[0]

In [ ]:
plt.figure(figsize=(8,4), dpi=200)
ax = sns.scatterplot(data=data_clean, x='USD_Production_Budget',
                     y='USD_Worldwide_Gross',
                     hue='USD_Worldwide_Gross',
                     size='USD_Worldwide_Gross')
ax.set(ylim=(0, 3000000000), xlim=(0, 4500000000)) 
plt.show()

In [ ]:
dt_index = pd.DatetimeIndex(data_clean.Release_Date)
decades = dt_index.year // 10 * 10
data_clean['Decade'] = decades
old_films = data_clean[data_clean.Decade <= 1960]
new_films = data_clean[data_clean.Decade > 1960]


In [ ]:
plt.figure(figsize=(8,4), dpi=200)
with sns.axes_style("whitegrid"):
    sns.regplot(data=old_films,
                x='USD_Production_Budget',
                y='USD_Worldwide_Gross',
                scatter_kws={'alpha': 0.4},
                line_kws={'color': 'black'})
    
plt.figure(figsize=(8,4), dpi=200)
with sns.axes_style("darkgrid"):
    ax = sns.regplot(data=new_films,
                     x='USD_Production_Budget',
                     y='USD_Worldwide_Gross',
                     color='#2f4b7c',
                     scatter_kws={'alpha': 0.3},
                     line_kws={'color': '#ff7c43'})

In [ ]:
regression = LinearRegression()
X = pd.DataFrame(new_films, columns=['USD_Production_Budget'])
y = pd.DataFrame(new_films, columns=['USD_Worldwide_Gross'])
regression.fit(X, y)
print(regression.intercept_)
print(regression.coef_)
print(regression.score(X, y))

In [25]:
budget_df = pd.DataFrame([[350000000]], columns=['USD_Production_Budget'])
revenue_estimate = regression.predict(budget_df)[0][0]
revenue_estimate = round(revenue_estimate, -6)